In [0]:
%sql
use catalog olist_project

##Revenue by state per month

In [0]:
%sql
SELECT 
    c.customer_state,
    DATE_TRUNC('month', o.order_purchase_timestamp) as month,
    SUM(p.payment_value) as revenue
FROM silver.orders o
JOIN silver.payments p ON o.order_id = p.order_id
JOIN silver.customers c ON o.customer_id = c.customer_id
GROUP BY customer_state, month
ORDER BY revenue DESC

In [0]:
from pyspark.sql.functions import desc,date_trunc,sum,col


order_df=spark.read.format("delta").table("olist_project.silver.orders")
# display(order_df)

payments_df=spark.read.format("delta").table("olist_project.silver.payments")
# payments_df.display()

customer_df=spark.read.format("delta").table("olist_project.silver.customers")
# customer_df.display()

rev_state_df=order_df.filter(col("order_status")=="delivered").join(payments_df, on='order_id', how='left').join(customer_df, on='customer_id', how='left').groupBy([col("customer_state"),date_trunc("month", col("order_purchase_timestamp")).alias("order_month")]).agg(sum("payment_value").alias("total_revenue")).orderBy(desc("total_revenue"))
# rev_state_df.display()
# Gold Table 1
rev_state_df.write.format("delta").mode("overwrite").saveAsTable("olist_project.gold.revenue_by_state")


Databricks data profile. Run in Databricks to view.

##2.Average delivery time by product category

In [0]:
delivery_df = spark.sql("""SELECT p.product_category_name,
                    avg(datediff(order_delivered_customer_date,order_purchase_timestamp)) as         avg_days_of_delivery
                    FROM olist_project.silver.orders o left join olist_project.silver.order_items ot on o.order_id = ot.order_id 
                    LEFT join olist_project.silver.products p 
                    ON ot.product_id = p.product_id
                    where o.order_status="delivered" AND o.order_delivered_customer_date IS NOT NULL
                    GROUP BY p.product_category_name
                    ORDER BY avg_days_of_delivery DESC""")
delivery_df.write.format("delta").mode("overwrite").saveAsTable("olist_project.gold.delivery_by_category")


##Late delivery flag table

In [0]:
from pyspark.sql.functions import col,when,count
order_df=spark.read.format("delta").table("olist_project.silver.orders")
order_df=order_df.withColumn("is_delayed",when(col("order_delivered_customer_date")>col("order_estimated_delivery_date"),1).otherwise(0))

customer_df=spark.read.format("delta").table("olist_project.silver.customers")

late_del_df=order_df.filter(col("order_status")=="delivered").join(customer_df, on='customer_id', how='left').groupBy(["customer_state"]).agg(sum("is_delayed").alias("late_orders"),count("order_id").alias("total_orders")).withColumn("delay_ratio",col("late_orders")/col("total_orders")).orderBy(desc("delay_ratio"))
# late_del_df.display()

# Gold Table 3
late_del_df.write.format("delta").mode("overwrite").saveAsTable("olist_project.gold.late_delivery_by_state")


In [0]:
for table in ["revenue_by_state", "delivery_by_category", "late_delivery_by_state"]:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM olist_project.gold.{table}").collect()[0][0]
    print(f"{table}: {count} rows")

In [0]:
spark.sql("""
SELECT customer_state, 
       late_orders,
       total_orders,
       ROUND(delay_ratio * 100, 2) as late_pct
FROM olist_project.gold.late_delivery_by_state
ORDER BY late_pct DESC
LIMIT 5
""").show()

In [0]:
spark.sql("""
SELECT 
    ROUND(AVG(delay_ratio) * 100, 2) as national_avg_late_pct,
    ROUND(MAX(delay_ratio) * 100, 2) as worst_state_pct,
    ROUND(MIN(delay_ratio) * 100, 2) as best_state_pct
FROM olist_project.gold.late_delivery_by_state
""").show()


In [0]:
spark.sql("OPTIMIZE olist_project.gold.revenue_by_state ZORDER BY (customer_state)")
spark.sql("OPTIMIZE olist_project.gold.delivery_by_category ZORDER BY (product_category_name)")
spark.sql("OPTIMIZE olist_project.gold.late_delivery_by_state ZORDER BY (delay_ratio)")

spark.sql("VACUUM olist_project.gold.revenue_by_state RETAIN 168 HOURS")
spark.sql("VACUUM olist_project.gold.delivery_by_category RETAIN 168 HOURS")
spark.sql("VACUUM olist_project.gold.late_delivery_by_state RETAIN 168 HOURS")
